In [ ]:
!export VLLM_USE_V1=0
!source ~/.bashrc

In [21]:
%load_ext autoreload
%autoreload 2

In [2]:
from helpers import json_to_llm_string
import random
import pandas as pd

In [ ]:

from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

In [ ]:
# 1. Initialize the LLM with the Qwen model name
# Use a specific model name from Hugging Face, for example 'Qwen/Qwen2.5-0.5B-Instruct'
# model_name = "Qwen/Qwen3-VL-32B-Instruct"
model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507"
llm = LLM(model=model_name, gpu_memory_utilization=0.8)

INFO 03-02 00:39:39 [utils.py:223] non-default args: {'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507'}
INFO 03-02 00:39:39 [model.py:529] Resolved architecture: Qwen3MoeForCausalLM
INFO 03-02 00:39:39 [model.py:1549] Using max model len 262144
INFO 03-02 00:39:40 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-02 00:39:40 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=67798) INFO 03-02 00:39:41 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantizati

(EngineCore_DP0 pid=67798) /home/anaya/Projects/deep-learning-26wi/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
(EngineCore_DP0 pid=67798)     Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
(EngineCore_DP0 pid=67798)     Minimum and Maximum cuda capability supported by this version of PyTorch is
(EngineCore_DP0 pid=67798)     (8.0) - (12.0)
(EngineCore_DP0 pid=67798)     
(EngineCore_DP0 pid=67798)   warnings.warn(


(EngineCore_DP0 pid=67798) INFO 03-02 00:39:42 [parallel_state.py:1234] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://192.168.1.11:50457 backend=nccl
(EngineCore_DP0 pid=67798) INFO 03-02 00:39:42 [parallel_state.py:1445] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=67798) INFO

(EngineCore_DP0 pid=67798) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore_DP0 pid=67798) [2026-03-02 00:39:45] WARNING _http.py:857: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading safetensors checkpoint shards:   0% Completed | 0/16 [00:00<?, ?it/s]


In [11]:
import pandas as pd

In [12]:
ag = pd.read_csv("../datasets/classified_interactions.tsv", delimiter="\t")

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "interaction_type": {
            "type": "string",
            "enum": list(ag["Top Label"].unique()) + ["No interaction"]
        }
    },
    "required": ["interaction_type"],
    "additionalProperties": False
}
structured_outputs_params = StructuredOutputsParams(json=schema)

# 3. Define sampling parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=128,
    structured_outputs=structured_outputs_params
)

In [21]:
x = """ 
        DRUG 1:
        "name": "Alteplase",
        "description": "Alteplase is a recombinant tissue plasminogen activator (rt-PA) used as a thrombolytic agent.[L43125] It cleaves plasminogen to form plasmin, an enzyme involved in the degradation of fibrin clots. In the absence of fibrin, the alteplase-mediated conversion of plasminogen is limited, thanks to the high affinity between alteplase and fibrin.[A252330,L43125] Alteplase is a purified glycoprotein of 527 amino acids expressed in Chinese hamster ovary (CHO) cells.[A252345,L43125] It was first approved by the FDA in 1987 for the management of thromboembolic disease, including acute myocardial infarction (AMI).[A252270] The use of alteplase to manage AMI has decreased thanks to the availability of safer treatments such as angioplasty and stenting. However, its use for the treatment of acute ischemic stroke (AIS) has increased over the years.[A252340] New thrombolytic agents derived from tissue plasminogen activator, such as [desmoteplase], [tenecteplase] and [reteplase], have also been developed.[A252270,A252345] Alteplase is also available as Cathflo Activase (intracatheter instillation) for the restoration of function to central venous access devices.[L34864]",
        "synthesis-reference": "Goeddel, DV., et al. (1988). Human tissue plasminogen activator (U.S. Patent No. 4,766,075 A). U.S. Patent and Trademark Office. https://patentimages.storage.googleapis.com/a7/38/45/ec3813293b05c5/US4766075.pdf",
        "indication": "Alteplase is indicated for the treatment of acute ischemic stroke (AIS) and for use in acute myocardial infarction (AMI) for the reduction of mortality and incidence of heart failure. Alteplase is also indicated for the lysis of acute massive pulmonary embolism, defined as acute pulmonary emboli obstructing blood flow to a lobe or multiple lung segments, and acute pulmonary emboli accompanied by unstable hemodynamics.[L43125]",
        "pharmacodynamics": "Alteplase binds to fibrin and plasminogen. Alteplase specificity for fibrin is achieved thanks to its high affinity for lysine residues. Also, it can bind plasminogen via loop structures called kringles, stabilized by three disulphide linkages similar to the ones in plasminogen. The specificity of alteplase for plasminogen bound to fibrin allows this drug to act in a clot- or fibrin-specific manner, leading to low concentrations of circulating plasmin and a lower risk of hemorrhagic transformation.[A252270,A252330] \r\n\r\nIn patients with acute myocardial infarction, alteplase reduces fibrinogen levels 3 to 6 hours after treatment. In patients with acute ischemic stroke, patients treated with alteplase have a significantly higher resolution of hyperdense artery sign, a marker of clot formation in the proximal middle cerebral artery, compared to those treated with placebo.[A252330] The use of alteplase increases the risk of bleeding and thromboembolic events. Rare cases of cholesterol embolism have also been reported.[L43125]",
        "mechanism-of-action": "Alteplase is a recombinant tissue plasminogen activator (rt-PA) that converts plasminogen to plasmin in a fibrin-dependent process. In the absence of fibrin, alteplase converts a limited amount of plasminogen. However, in the presence of fibrin clots, alteplase binds to fibrin and cleaves the arginine-valine bond at positions 560 and 561 of plasminogen, converting it into its active form, plasmin. Plasmin in turn degrades the fibrin matrix of the thrombus and promotes clot dissolution.[A252270] Alteplase initiates local fibrinolysis with limited systemic proteolysis.[L43125]",
        "toxicity": "Toxicity information regarding alteplase is not readily available. Patients experiencing an overdose are at an increased risk of severe adverse effects such as risk of bleeding and thromboembolic events.[L43125] Symptomatic and supportive measures are recommended. The carcinogenic potential of alteplase or its effect on fertility have not been evaluated. _In vivo_ studies evaluating tumorigenicity and _in vitro_ studies evaluating mutagenicity were negative.[L43125] It has been estimated that the acute oral and dermal toxicity of alteplase is above 5,000 mg/kg.[L43175]",
        "metabolism": "Alteplase is mainly metabolized by the liver. The carbohydrate and polypeptide domains of alteplase interact with hepatic glycoprotein receptors, leading to receptor-mediated endocytosis.[A252270] _In vivo_ studies suggest that alteplase follows zero-order kinetics, meaning that its metabolism is saturable at higher plasma concentrations.[A252270]",
        "absorption": "Healthy volunteers with a baseline endogenous tissue plasminogen activator (t-PA) of 3.3 ng/ml had a 290-fold increase in baseline concentrations after receiving alteplase at an infusion rate of 0.25 mg/kg for 30 min; with an infusion rate of 0.5 mg/kg, a 550-fold increase was observed.[A252270] Acute myocardial infarction patients (n=12) given 10 mg of alteplase in a 2-minute infusion reached a peak plasma concentration of 3310 ng/ml. This was followed by 50 mg of alteplase in 1 h and 30 mg in 1.5 h, resulting in steady-state plasma levels of 2210 ng/ml and 930 ng/ml, respectively.[A252285]",
        "half-life": "Alteplase has an initial half-life of less than 5 minutes in patients with acute myocardial infarction (AMI). The dominant initial plasma half-life of the 3-hour and the accelerated regimens for AMI are similar.[L43125]",
        "protein-binding": "Not available.",
        "route-of-elimination": "In healthy volunteers, more than 80% of alteplase is eliminated through urine 18 hours after administration.[A252270]",
        "volume-of-distribution": "The initial volume of distribution approximates plasma volume.[L43125] The average volume of distribution of the central compartment goes from 3.9 to 4.3 L, and the volume of distribution at steady state goes from 7.2 to 12 L.[A252270]",
        "clearance": "Alteplase has a plasma clearance between 380 and 570 mL/min.[L43125]"
        
        DRUG 2:
        "name": "Peginterferon alfa-2a",
        "description": "Peginterferon alfa-2a is a form of recombinant interferon used as part of combination therapy to treat chronic Hepatitis C, an infectious liver disease caused by infection with Hepatitis C Virus (HCV). HCV is a single-stranded RNA virus that is categorized into nine distinct genotypes, with genotype 1 being the most common in the United States, and affecting 72% of all chronic HCV patients [L852]. Treatment options for chronic Hepatitis C have advanced significantly since 2011, with the development of Direct Acting Antivirals (DAAs) resulting in less use of Peginterferon alfa-2a. Peginterferon alfa-2a is derived from the alfa-2a moeity of recombinant human interferon and acts by binding to human type 1 interferon receptors. Activation and dimerization of this receptor induces the body's innate antiviral response by activating the janus kinase/signal transducer and activator of transcription (JAK/STAT) pathway. Use of Peginterferon alfa-2a is associated with a wide range of severe adverse effects including the aggravation and development of endocrine and autoimmune disorders, retinopathies, cardiovascular and neuropsychiatric complications, and increased risk of hepatic decompensation in patients with cirrhosis. The use of Peginterferon alfa-2a has largely declined since newer interferon-free antiviral therapies have been developed.\r\n\r\nIn a joint recommendation published in 2016, the American Association for the Study of Liver Diseases (AASLD) and the Infectious Diseases Society of America (IDSA) no longer recommend Peginterferon alfa-2a for the treatment of Hepatitis C [A19593]. Peginterferon alfa-2a was used alongside [DB00811] with the intent to cure, or achieve a sustained virologic response (SVR), after 48 weeks of therapy. SVR and eradication of HCV infection is associated with significant long-term health benefits including reduced liver-related damage, improved quality of life, reduced incidence of Hepatocellular Carcinoma, and reduced all-cause mortality [A19626].\r\n\r\nPeginterferon alfa-2a is available as a fixed dose injector (tradename Pegasys) used for the treatment of chronic Hepatitis C. Approved in 2002 by the FDA, Pegasys is indicated for the treatment of HCV with [DB00811] or other antiviral drugs [FDA Label]. When combined together, Peginterferon alfa-2a and [DB00811] have been shown to achieve a SVR between 36% for genotype 1 and 59% for genotypes 2-6 after 48 weeks of treatment.",
        "synthesis-reference": null,
        "indication": "Peginterferon alfa-2a is indicated for the treatment of HCV in combination with other antiviral drugs in patients over 5 years of age with compensated liver disease [FDA Label]. May be used as a monotherapy in patients with contraindications to or significant intolerance to other anti-viral therapies.\r\n\r\nPeginterferon alfa-2a is also indicated as a monotherapy for adult patients with HBeAg positive and HBeAg negative chronic hepatitis B infection who have compensated liver disease and evidence of viral replication and liver inflammation [FDA Label].",
        "pharmacodynamics": "Peginterferon alfa-2a induces the body's innate antiviral response [FDA Label].",
        "mechanism-of-action": "Peginterferon alfa-2a is derived from recombinant human interferon's alfa-2a moeity [FDA Label]. It binds to and activates human type 1 interferon receptors causing them to dimerize. This activates the JAK/STAT pathway. Activation of the JAK/STAT pathway increases expression of multiple genes in multiple tissues involved in the innate antiviral response.",
        "toxicity": "Peginterferon alfa-2a may manifest neuropsychiatric complications include suicide, suicidal ideation, homicidal ideation, depression, relapse of drug addiction, and drug overdose [FDA Label]. Hypertension, supraventricular arrhythmias, chest pain, and myocardial infarction have been observed in patients using Peginterferon alfa-2a. Peginterferon alfa-2a may produce myelosuppression as well as the development or aggravation of autoimmune disorders including myositis, hepatitis, thrombotic thrombocytopenic purpura, idiopathic thrombocytopenic purpura, psoriasis, rheumatoid arthritis, interstitial nephritis, thyroiditis, and systemic lupus erythematosus. Peginterferon alfa-2a causes or aggravates hypothyroidism and hyperthyroidism. Hyperglycemia, hypoglycemia, and diabetes mellitus have been observed to develop in patients treated with Peginterferon alfa-2a. Peginterferon alfa-2a may decrease or produce loss of vision, retinopathy including macular edema, retinal artery or vein thrombosis, retinal hemorrhages and cotton wool spots, optic neuritis, papilledema and serous retinal detachment. Peginterferon mayy be related to increased ischemic and hemorrhagic cerebrovascular events. Patients with cirrhosis on Peginterferon alfa-2a are at risk of hepatic decompensation. Dyspnea, pulmonary infiltrates, pneumonia, bronchiolitis obliterans, interstitial pneumonitis, pulmonary hypertension and sarcoidosis may be induced or aggravated by Peginterferon alfa-2a. Serious and severe infections (bacterial, viral, or fungal) have been reported during treatment with Peginterferon alfa-2a. Ulcerative and hemorrhagic/ischemic colitis have been observed within 12 weeks of starting Peginterferon alfa-2a treatment. Pancreatitis and peripheral nephropathy have also been reported. Peginterferon alfa-2a is associated with growth inhibition in pediatric patients. Use of Peginterferon alfa-2a while pregant may result in delopmental abnormalities or death of the fetus.",
        "metabolism": null,
        "absorption": "Peginterferon alfa-2a reaches peak plasma concentration 72-96 hours after subcutaneous administration [FDA Label]. Trough concentrations at week 48 are approximately 2 fold higher than week 1. The peak to trough ratio at week 48 is 2.",
        "half-life": "The mean terminal half-life of peginterferon alfa-2a is 164 in a range of 84-353 hours [FDA Label].",
        "protein-binding": null,
        "route-of-elimination": null,
        "volume-of-distribution": null,
        "clearance": "The mean systemic clearance of peginterferon alfa-2a is 94 milliliters per hour [FDA Label]."
        """



In [31]:
# 4. Generate the responses
outputs = llm.generate([x], sampling_params)

# 5. Print the results
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: ' \n        DRUG 1:\n        "name": "Alteplase",\n        "description": "Alteplase is a recombinant tissue plasminogen activator (rt-PA) used as a thrombolytic agent.[L43125] It cleaves plasminogen to form plasmin, an enzyme involved in the degradation of fibrin clots. In the absence of fibrin, the alteplase-mediated conversion of plasminogen is limited, thanks to the high affinity between alteplase and fibrin.[A252330,L43125] Alteplase is a purified glycoprotein of 527 amino acids expressed in Chinese hamster ovary (CHO) cells.[A252345,L43125] It was first approved by the FDA in 1987 for the management of thromboembolic disease, including acute myocardial infarction (AMI).[A252270] The use of alteplase to manage AMI has decreased thanks to the availability of safer treatments such as angioplasty and stenting. However, its use for the treatment of acute ischemic stroke (AIS) has increased over the years.[A252340] New thrombolytic agents derived from tissue plasminogen activat

In [9]:
SYSTEM_PROMPT = """You are an expert in pharmaceutical drugs.
Your role is to determine how two classify pairs of drugs into
an interaction type, or state that there is "No interaction".
You do not need to provide any reasoning, but your decision
should be grounded in scientific facts and drug information
that is presented to you.
"""

BASE_MESSAGES = [
    {"role": "system", "content": SYSTEM_PROMPT}
]



In [4]:
from enum import Enum

In [ ]:
from pydantic import BaseModel

class InteractionPrediction(BaseModel)

In [48]:
print(json_to_llm_string({"hello" : ["how", "are", "you"], "nest" : { "one": {"sup": "hi"},}}))

hello:
  - how
  - are
  - you
nest:
  one:
    sup: hi


In [5]:
from openai import OpenAI

In [6]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token-abc123",
)

completion = client.chat.completions.create(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    messages=[
        {"role": "user", "content": "Hello!"},
    ],
)

In [8]:
completion.choices[0].message.content

'Hello! How can I assist you today? 😊'

In [15]:
schema = {
    "type": "object",
    "properties": {
        "interaction_type": {
            "type": "string",
            "enum": list(ag["Top Label"].unique()) + ["No interaction"]
        }
    },
    "required": ["interaction_type"],
    "additionalProperties": False
}

In [23]:
texts = [
    "The team won the championship.",
    "The president signed a new bill.",
    "New GPU architecture released."
]

# build batch prompts
messages_batch = []
for text in texts:
    messages_batch.append(BASE_MESSAGES + [{"role": "user", "content": text}])

print(messages_batch)


messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": x}
]

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-27B-Instruct",
    messages=messages,
    temperature=0,
    max_tokens=32,
    response_format={
        "type": "json_schema",
        "json_schema": schema
    }
)


[[{'role': 'system', 'content': 'You are an expert in pharmaceutical drugs.\nYour role is to determine how two classify pairs of drugs into\nan interaction type, or state that there is "No interaction".\nYou do not need to provide any reasoning, but your decision\nshould be grounded in scientific facts and drug information\nthat is presented to you.\n'}, {'role': 'user', 'content': 'The team won the championship.'}], [{'role': 'system', 'content': 'You are an expert in pharmaceutical drugs.\nYour role is to determine how two classify pairs of drugs into\nan interaction type, or state that there is "No interaction".\nYou do not need to provide any reasoning, but your decision\nshould be grounded in scientific facts and drug information\nthat is presented to you.\n'}, {'role': 'user', 'content': 'The president signed a new bill.'}], [{'role': 'system', 'content': 'You are an expert in pharmaceutical drugs.\nYour role is to determine how two classify pairs of drugs into\nan interaction ty

BadRequestError: Error code: 400 - {'error': {'message': '6 validation errors:\n  {\'type\': \'missing\', \'loc\': (\'body\', \'response_format\', \'function-wrap[__log_extra_fields__()]\', \'json_schema\', \'name\'), \'msg\': \'Field required\', \'input\': {\'type\': \'object\', \'properties\': {\'interaction_type\': {\'type\': \'string\', \'enum\': [\'Vascular haemorrhagic disorders\', \'Gastrointestinal haemorrhages NEC\', \'Product quality, supply, distribution, manufacturing and quality system issues\', \'Haematological and lymphoid tissue therapeutic procedures\', \'Increased intracranial pressure and hydrocephalus\', \'Central nervous system vascular disorders\', \'Coagulopathies and bleeding diatheses (excl thrombocytopenic)\', \'Platelet disorders\', \'Exposures, chemical injuries and poisoning\', \'Haematopoietic neoplasms (excl leukaemias and lymphomas)\', \'Haemoglobinopathies\', \'Vitamin related disorders\', \'Immunodeficiency syndromes\', \'Allergic conditions\', \'Bacte